In [0]:
dbutils.widgets.text("business_date", "", "Business Date (YYYY-MM-DD, empty=latest)")

import sys
sys.path.append("/Workspace/Shared")
import aws_utils

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, when, row_number, trim
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

env = aws_utils.get_env_prefix()

business_date_param = dbutils.widgets.get("business_date").strip()

if business_date_param:
    run_date = business_date_param
else:
    run_date = str(
        spark.table(f"`{env}-cdh`.primary.exposure")
        .filter(col("SourceSystem") == "profits")
        .agg({"BusinessDate": "max"})
        .collect()[0][0]
    )

run_date_lit = lit(run_date).cast("date")

print(f"env:       {env}")
print(f"run_date:  {run_date}")

# dtransfer_VatSettlement

Replicates on-prem stored procedure `[PLTFRM].[dtransfer].[prep_VatSettlement]`,
extended (DNA26-1804) to include **agreement-level** settlements in a dedicated
`AgreementNumber` column.

Returns settlements in the IT perimeter with the customer's tax number.

## Table Mapping

| On-Prem | Cloud | Notes |
| --- | --- | --- |
| `analytic.bi.Settlement` | `{env}-cdh.derivative.dsettlementexposure` | Filter: **`isPromiseActive = 1`** (IT `TABLE_STATUS='1'` perimeter, interim — see spec/steering) AND (`ExposureKey IS NOT NULL` OR `AgreementKey IS NOT NULL`). Loan rows: `AccountNumber = ExposureKey`. Agreement rows (CASE_ACC_SYSTEM=19): `AgreementNumber = AgreementKey` (dash-composite = on-prem `ClcCase.CASE_ACC_NUMBER`). No COALESCE, no synthesized key. |
| `DWH.prt.Party` | `{env}-cdh.primary.party` | For TaxRegistrationNumber via CDI |
| `PRC.ref.partyTaxNumber` (INNER JOIN) | `party.TaxRegistrationNumber` via **INNER** join on CDI | On-prem INNER-joins the tax table, so settlements with no resolvable tax number are dropped (never NULL). Applies to agreement rows too (they carry a CDI). |
| `DWH.ref.Portfolio` | `{env}-cdh.primary.portfolio` | PortfolioId lookup |

> **Option 1 (DNA26-1804):** `AccountNumber = ExposureKey` (loan), `AgreementNumber = AgreementKey` (agreement), mutually exclusive per row. QC→QCA 1→N migrated settlements have an `ExposureKey` and no `AgreementKey`, so they flow through as **loan rows** (no synthesized agreement key).
> **Banking portfolios ALWAYS excluded** (unconditional): 31 Core Retail, 32 Core Wholesale, 60 ARTEMIS/Attica, 92 JEWEL, 87 GALENE.
> **Perimeter flag:** `isPromiseActive` (NOT `IsActiveFlag`) — interim decision pending IT; see `.kiro/steering/settlement-debtor-perimeter.md`.

## Output

- `{env}-cdh-delivery.it.dtransfer_vatsettlement` — daily overwrite. Columns:
  `AccountNumber, AgreementNumber, SettlementType, SettlementCode, PortfolioId, taxNumber, datadate`.

In [0]:
# ==================== Load Source Tables ====================

# Banking portfolios are ALWAYS excluded (unconditional, DNA26-1804):
#   31 Core Retail, 32 Core Wholesale, 60 ARTEMIS/Attica, 92 JEWEL, 87 GALENE.
BANKING_PORTFOLIO_IDS = [31, 32, 60, 92, 87]

# Settlements in the IT perimeter (Option 1, DNA26-1804).
#
# DRIVE from curated primary.settlementexposure: it carries the natural loan vs
# agreement split (a profits agreement settlement, CASE_ACC_SYSTEM=19, has
# ExposureKey NULL + AgreementKey populated; loan settlements have ExposureKey).
#   AccountNumber   = ExposureKey  (loan rows; agreement rows -> NULL)
#   AgreementNumber = AgreementKey (agreement rows; loan rows -> NULL)
# No COALESCE, no synthesized key. QC->QCA 1->N settlements have an ExposureKey
# (no AgreementKey) so they naturally become loan rows.
#
# Perimeter uses isPromiseActive (IT ClcPayPromises.TABLE_STATUS='1' +
# CASE_ACC_SYSTEM IN (4,19)), NOT IsActiveFlag. That flag lives on the derivative,
# so we LEFT-SEMI join dsettlementexposure(isPromiseActive=1) to gate the perimeter
# while keeping the curated loan/agreement keys. (Curated keys match on-prem
# exactly; the derivative fills both keys so it cannot drive the split.)
promise_active = (
    spark.table(f"`{env}-cdh`.derivative.dsettlementexposure")
    .filter((col("BusinessDate") == run_date_lit) & (col("isPromiseActive") == 1))
    .select("SettlementKey", "SourceSystem")
    .distinct()
)

settlement = (
    spark.table(f"`{env}-cdh`.primary.settlementexposure")
    .filter(
        (col("BusinessDate") == run_date_lit)
        & ((col("ExposureKey").isNotNull()) | (col("AgreementKey").isNotNull()))
    )
    .join(promise_active, on=["SettlementKey", "SourceSystem"], how="left_semi")
    .select(
        # agreement settlement = ExposureKey NULL & AgreementKey present
        when(col("ExposureKey").isNull() & col("AgreementKey").isNotNull(), lit(None))
        .otherwise(col("ExposureKey")).alias("AccountNumber"),
        when(col("ExposureKey").isNull() & col("AgreementKey").isNotNull(), col("AgreementKey"))
        .otherwise(lit(None)).alias("AgreementNumber"),
        col("SettlementType"),
        col("SettlementKey").cast("string").alias("SettlementCode"),
        col("PortfolioName"),
        col("CDI"),
    )
)

# Portfolio lookup for PortfolioId
portfolio = (
    spark.table(f"`{env}-cdh`.primary.portfolio")
    .filter(col("BusinessDate") == run_date_lit)
    .select("PortfolioId", "PortfolioName")
)

# Party for TaxRegistrationNumber: run-date snapshot, tax-first dedup. Kept
# IDENTICAL in scope and ordering to dtransfer_Customer.party_tax and
# portfolio_cdi_bal.party_tax (review #3/#10): non-blank tax first, source
# priority, latest date, then TaxRegistrationNumber as a deterministic tie-break.
party_all = spark.table(f"`{env}-cdh`.primary.party").filter(col("BusinessDate") == run_date_lit)

w = Window.partitionBy("CDI").orderBy(
    # Prefer a party row that actually has a tax number: for a given CDI, some
    # source rows carry a NULL TaxRegistrationNumber while a sibling row carries
    # one. Rank non-null tax first so the dedup winner has a usable taxNumber.
    when(trim(col("TaxRegistrationNumber")) != "", 0).otherwise(1),
    when(col("SourceSystem") == "profits", 1)
    .when(col("SourceSystem") == "qca", 2)
    .otherwise(3),
    col("BusinessDate").desc(),
    col("TaxRegistrationNumber")
)

party = (
    party_all
    .withColumn("_rn", row_number().over(w))
    .filter(col("_rn") == 1)
    .drop("_rn")
    .select("CDI", when(trim(col("TaxRegistrationNumber")) != "", trim(col("TaxRegistrationNumber"))).alias("taxNumber"))
)

# ==================== Build VatSettlement ====================

vatsettlement_df = (
    settlement
    .join(portfolio, on="PortfolioName", how="inner")
    # Banking portfolios ALWAYS excluded (unconditional, DNA26-1804).
    .filter(~col("PortfolioId").isin(*BANKING_PORTFOLIO_IDS))
    # INNER join on party: mirrors on-prem prep_VatSettlement, which INNER JOINs
    # PRC.ref.partyTaxNumber and so drops any settlement whose customer has no tax
    # number. Applies to agreement rows too (they carry a CDI). Post QC-decommission,
    # some settlements lost their customer/CDI linkage; those rows cannot resolve a
    # taxNumber and are excluded here instead of being emitted as NULL.
    .join(party, on="CDI", how="inner")
    # Guarantee the NOT NULL contract: the INNER join only ensures the CDI exists
    # in party; the dedup winner can still carry a NULL TaxRegistrationNumber when
    # no party row for that CDI has a tax number. Drop those rows so taxNumber is
    # never NULL, matching on-prem's INNER JOIN on PRC.ref.partyTaxNumber.
    .filter(col("taxNumber").isNotNull())
    .select(
        col("AccountNumber"),
        col("AgreementNumber"),
        col("SettlementType"),
        col("SettlementCode"),
        col("PortfolioId"),
        col("taxNumber"),
        run_date_lit.alias("datadate"),
    )
    .distinct()
)

# Post-build assertion: enforce the NOT NULL contract rather than assume it.
assert vatsettlement_df.filter(col("taxNumber").isNull()).limit(1).count() == 0, \
    "dtransfer_vatsettlement produced NULL taxNumber rows (NOT NULL contract violated)"

# Every row must carry exactly ONE of AccountNumber / AgreementNumber.
_both_or_neither = vatsettlement_df.filter(
    (col("AccountNumber").isNull() & col("AgreementNumber").isNull())
    | (col("AccountNumber").isNotNull() & col("AgreementNumber").isNotNull())
).limit(1).count()
assert _both_or_neither == 0, \
    "dtransfer_vatsettlement: AccountNumber/AgreementNumber must be mutually exclusive"

vat_count = vatsettlement_df.count()
print(f"dtransfer_vatsettlement rows: {vat_count:,}")

# ==================== Pre-write guards ====================
# Empty-result guard: raise (do NOT silently overwrite with 0 rows).
if vat_count == 0:
    raise ValueError("dtransfer_vatsettlement produced 0 rows — aborting (empty-result guard)")

# Day-over-day row-count floor: fail if the new count drops >25% vs the currently
# delivered table (guards against upstream breakage). Skipped on first load.
target_vatsettlement = f"`{env}-cdh-delivery`.it.dtransfer_vatsettlement"
if spark.catalog.tableExists(target_vatsettlement):
    prev_count = spark.table(target_vatsettlement).count()
    if prev_count > 0 and vat_count < prev_count * 0.75:
        raise ValueError(
            f"dtransfer_vatsettlement row-count floor breached: {vat_count:,} < 75% of "
            f"previous {prev_count:,}. Investigate before overwriting."
        )

In [0]:
# ==================== Write dtransfer_vatsettlement ====================

target_vatsettlement = f"`{env}-cdh-delivery`.it.dtransfer_vatsettlement"

(
    vatsettlement_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_vatsettlement)
)

print(f"Written to {target_vatsettlement}")